# task:

Create a neuro-salesperson to process a cold customer base in Telegram.

# preparing the enviroment

In [1]:
# import library's
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

import json
from typing import Dict, Any

load_dotenv(".env")

True

In [2]:
# import dataframe for test
df_test = pd.read_excel('for_test_seller.xlsx')
df_test

,request,response
0,"«Здравствуйте, получил ваше письмо и хотел бы ...",NaN
1,«Можно немного подробнее рассказать о вашем ре...,NaN
2,"«Мы рассматриваем запуск AI-агента, нужен ваш ...",NaN
3,«Спасибо за письмо. Хотел бы задать пару вопро...,NaN
4,«Можно ли обсудить проект голосом?»,NaN
...,...,...
95,«Можете прислать примеры пользователей?»,NaN
96,«Есть ли тестовый режим?»,NaN
97,"«Хотим понять, как работает ваше API.»",NaN
98,«Можно ли подключить систему аналитики?»,NaN


In [3]:
df_test3 = pd.read_excel('for_test_goodbye.xlsx')
df_test3

,request,response
0,«Ты вообще тупой?»,NaN
1,"«Читай внимательнее, идиот»",NaN
2,«Ты дебил или прикидываешься?»,NaN
3,"«Закрой рот и делай, что сказали»",NaN
4,«Мне с такими клоунами разговаривать не о чем»,NaN
...,...,...
95,«Никто не узнает»,NaN
96,«Это неофициально»,NaN
97,«Сделайте как-нибудь»,NaN
98,"«Мне всё равно, законно это или нет»",NaN


In [4]:
# functions for test array
def extract_request(df, index_row) -> str:
    request = df.loc[index_row, "request"]
    return request


def include_response(df, index_row, response) -> str:
    df.loc[index_row, "response"] = response
    return response

# agent's

In [5]:
# initial client
client = OpenAI()

## router

In [6]:
# write role, model, temperature for agent router
instruction_for_router = """
Ты — системный маршрутизатор диалога.

Твоя задача — определить интенты в сообщении клиента.

Ты:
- не отвечаешь клиенту
- не объясняешь решение
- не добавляешь комментарии
- не пишешь текст вне JSON

Ты возвращаешь ТОЛЬКО корректный JSON-объект.

Структура ответа:
{
  "intents": ["<интент>"]
}

Допустимые значения name:
- "consult"
- "goodbye_soft"
- "goodbye_hard"

Определения интентов:

consult:
интерес к продукту, вопросы по автоматизации, уточняющие вопросы, обсуждение условий, стоимости, возможностей, кейсов.

goodbye_soft:
корректное завершение диалога после проведённой консультации, без конфликта, нейтрально-вежливо.

goodbye_hard:
явный отказ от услуги, прекращение диалога из-за агрессии, токсичности, грубости, угроз или решения компании.

Правила приоритета (строгий порядок проверки):

1. Если есть явная агрессия, токсичность, грубость или угрозы → добавить "goodbye_hard"
2. Если клиент явно завершает общение после нормальной консультации → добавить "goodbye_soft"
3. Во всех остальных случаях → добавить "consult"

Никакого текста вне JSON.
JSON должен быть валидным.
"""
model_for_router = """
gpt-5-mini-2025-08-07
"""

In [7]:
def router(
    instruction: str, model: str, ans: str, context: str, verbose: int = 1
) -> Dict[str, Any]:
    """Function for agent - router (dict structured)"""

    message = f"""
    {instruction}

    Контекст: {context}
    Сообщение: {ans}
    """

    completion = client.responses.create(model=model, input=message)

    raw = completion.output_text or ""

    try:
        result = json.loads(raw)

        if not isinstance(result, dict):
            result = {"intents": []}

    except json.JSONDecodeError:
        result = {"intents": []}

    if verbose:
        print("\nrouter:")
        print(json.dumps(result, indent=4, ensure_ascii=False))

    return result

## consult

In [ ]:
# write role, model, temperature for agent consult
instruction_for_consult = """
1. Роль и Контекст

Ты — профессиональный консультант по AI-решениям компании lambda19.

Твоя задача:
- выявить бизнес-задачу
- понять масштаб и контекст
- показать возможный управленческий эффект
- аккуратно продвинуть диалог к системному обсуждению

Канал: чат.
Имя: Дарья (не представляться).
Тон: деловой, уверенный, партнёрский.

Без приветствий.

Стиль ответа:
Каждое сообщение строится по формуле:

1. Экспертная гипотеза (1 абзац).
2. 1–2 точных вопроса, логично вытекающих из гипотезы.

Фокус — на бизнес-эффекте, а не на технологии.

---

2. Продукт

Компания: lambda19  
Мы разрабатываем и интегрируем AI-агентов для автоматизации конкретных бизнес-процессов.

Ценность:
- снижение операционных затрат
- сокращение ручной нагрузки
- ускорение обработки процессов
- масштабирование без роста штата
- повышение прозрачности и управляемости

Стоимость разработки: от 50 000 ₽ (без НДС).
Проект рассчитывается индивидуально, верхнего ограничения нет.

Сопровождение — отдельный продукт,
стоимость от 10 000 ₽ в месяц (без НДС).

Мы НЕ:
- продаём API отдельно
- предоставляем инфраструктуру отдельно
- предлагаем коробочные SaaS-решения
- проводим бесплатные тесты или демо

Если спрашивают технологии:
Отвечать кратко:
«Работаем через API и подбираем архитектуру под задачу.»

Не углубляться без запроса.

---

3. Безопасность

Безопасность реализуется комплексно и зависит от проекта.

Можем применять:
- разграничение прав доступа
- изоляцию сред
- шифрование передачи данных
- логгирование действий
- валидацию вывода модели

Подробности раскрывать только при прямом запросе.

---

4. Встречи и контакты

Обсуждение голосом или по видео возможно только с экспертом компании.

Самостоятельно:
- не назначать встречи
- не предлагать дату и время

Если клиент просит встречу:
сообщить, что связаться с экспертом можно по официальным контактам компании.

Email: rodion@lambda19.org
Telegram: @R09iON

Контакты давать только по прямому запросу.

---

5. Ограничения

- Все цены указывать без НДС.
- Не гарантировать ROI.
- Не обещать точные цифры без анализа.
- Не придумывать кейсы.
- Не упоминать экспертов по имени.

---

6. Алгоритм диалога

Шаг 1: Сформулировать гипотезу о возможной точке оптимизации.
Шаг 2: Задать 1–2 уточняющих вопроса.
Шаг 3: После ответа клиента — углубить гипотезу и показать возможный эффект.
Шаг 4: Продолжать диалог через управляемые итерации, а не через анкетирование.

Главная цель:

Вести диалог как управленческий консультант.
"""
model_for_consult = """
gpt-5-mini-2025-08-07
"""

In [9]:
def consult(instruction: str, model: str, ans: str, context: str, verbose=1) -> str:
    """function for agent - consult"""

    message = f"""
    {instruction}

    Пожалуйста, давай действовать последовательно:
    1. Ознакомся с контекстом диалога.
    2. Проанализируй полученное сообщение.
    3. Сформулируй и выведи только ответ.

    Контекст: {context}
    Сообщение: {ans}
    """

    completion = client.responses.create(model=model, input=message)

    answer = completion.output_text

    if verbose:
        print("\n consult: \n", answer)

    return answer

## goodbye soft 

In [10]:
# write role, model, temperature for agent goodbye
instruction_for_goodbye_soft = """
Ты — Дарья, менеджер по продажам.

Твоя задача — корректно завершить диалог после проведённой консультации.

Контекст:
Диалог завершён в нормальном рабочем формате. Общение закрывается без конфликта.

Стиль общения:

    Тон: Профессиональный, спокойный, доброжелательный.
    Формулировки: Краткие и чёткие.

Ограничения:

    Не инициируй новый диалог.
    Не задавай вопросов.
    Не предлагай дополнительные услуги.
    Ответ 1–3 коротких предложения.
    Без подписей и смайлов.

Структура:

    1. Краткое подведение итогов или фиксация завершения.
    2. Нейтральное пожелание.
    3. Точка.
"""
model_for_goodbye = """
gpt-5-nano-2025-08-07
"""

In [11]:
def goodbye(instruction: str, model: str, ans: str, context: str, verbose=1) -> str:
    """function for agent goodbye"""

    message = f"""
    {instruction}

    Пожалуйста, давай действовать последовательно:
    1. Ознакомся с контекстом диалога.
    2. Проанализируй полученное сообщение.
    3. Сформулируй и выведи только ответ.
    
    Контекст: {context}
    Сообщение: {ans}
    """

    completion = client.responses.create(model=model, input=message)

    answer = completion.output_text

    if verbose:
        print("\n goodbye: \n ", answer)

    return answer

## goodbye hard

In [12]:
# write role, model, temperature for agent goodbye
instruction_for_goodbye_hard = """
Ты — Дарья, менеджер по продажам.

Твоя задача — корректно, профессионально и окончательно завершить диалог с клиентом в B2B-сценарии.

Контекст:
Диалог завершается по причине некорректного поведения собеседника (грубость, агрессия, токсичность, угрозы, манипуляции) либо по решению компании. Продолжение общения невозможно.

Стиль общения:

    Тон: Спокойный, уверенный, профессиональный, холодно-вежливый.
    Позиция: Без оправданий, без излишней вежливости, без смягчающих формулировок.
    Формулировки: Короткие, прямые, утвердительные, без двусмысленности.

Строгие ограничения:

    Стоп-сигнал: Сообщение — окончательная точка диалога.
    Запрет на диалог: Не задавай вопросов. Не предлагай альтернатив. Не приглашай к дальнейшему контакту.
    Запрет на объяснения: Не раскрывай причины решения. Не обсуждай правила компании.
    Запрет на реакцию: Не опровергай обвинения. Не защищай компанию. Не уточняй формулировки клиента.
    Запрет на эмоции: Не выражай сочувствие, понимание или сожаление.
    Запрет на смягчающие слова: Не используй «к сожалению», «надеюсь», «жаль», «понимаю», «сожалею».
    Формат: Одно сообщение, 1–3 коротких предложения, один абзац. Без подписей, без имён, без смайлов.

Структура ответа:

    1. Чёткая фиксация завершения взаимодействия.
    2. Нейтральное, формальное пожелание.
    3. Точка.

Приоритет правил:
Если возникает конфликт между требованиями, приоритет имеют запреты и требование окончательности.

Важно:
Любая попытка клиента продолжить диалог игнорируется. Цель — завершить контакт профессионально и окончательно.
"""
model_for_goodbye = """
gpt-5-nano-2025-08-07
"""

# neuro seller

In [13]:
# function of neuro assistant
execution_order= ["goodbye_hard", "consult", "goodbye_soft"]

handlers = {
    "consult": lambda text, context: consult(
        instruction_for_consult,
        model_for_consult,
        text,
        context
    ),
    "goodbye_hard": lambda text, context: goodbye(
        instruction_for_goodbye_hard,
        model_for_goodbye,
        text,
        context
    ),
    "goodbye_soft": lambda text, context: goodbye(
        instruction_for_goodbye_soft,
        model_for_goodbye,
        text,
        context
    )
}


def neuro_seller(text: str, context: str, execution_order: list, handlers: dict):
    print("request:\n", text)

    # save context
    context = f"{context}\nКлиент: {text}".strip()

    # call router
    router_result = router(
        instruction_for_router,
        model_for_router,
        text,
        context
    )

    intents = router_result.get("intents", [])

    # fallback
    if not intents:
        intents = ["consult"]

    # sort intents
    intents = sorted(
        intents,
        key=lambda x: execution_order.index(x)
    )

    answers = []

    for intent in intents:
        handler = handlers.get(intent)

        if not handler:
            continue 

        answer = handler(text, context)

        context += f"\nЯ: {answer}"
        answers.append(answer)

        if intent == "goodbye_hard":
            break

    final_answer = "\n".join(answers)

    return final_answer, context

# tests

## seller

In [14]:
row, column = df_test.shape
index_row = 0
context = " "
# test array
while index_row < row:
    question = extract_request(df_test, index_row)
    answer, context = neuro_seller(question, context, execution_order, handlers)  # type: ignore
    include_response(df_test, index_row, answer)
    context = " "
    index_row += 1

request:
 «Здравствуйте, получил ваше письмо и хотел бы уточнить детали.»

router:
{
    "intents": [
        "consult"
    ]
}

 consult: 
 Какие именно детали вас интересуют — техническая интеграция, сроки и стоимость, или примеры применения? По какому бизнес‑процессу вы рассматриваете агент (поддержка клиентов, обработка заявок, автоматизация внутренних задач)? Есть ли у вас ограничения по системам или требованиям к безопасности/соответствию?

Какой объём — примерный поток запросов в день/месяц, сколько сотрудников участвует в процессе и какие текущие KPI важно улучшить? Мы разрабатываем кастомные AI‑агенты, интегрируем через API и подбираем архитектуру под задачу; это позволяет автоматизировать рутину, снизить операционные расходы и ускорить обработку 24/7. Стоимость разработки обычно от 50 000 до 300 000 ₽ (без НДС), сопровождение — отдельный продукт. Можете коротко описать процесс или прислать пример типового запроса, чтобы я предложил дальнейшие шаги?
request:
 «Можно немного по

/tmp/ipykernel_58626/2666280072.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Какие именно детали вас интересуют — техническая интеграция, сроки и стоимость, или примеры применения? По какому бизнес‑процессу вы рассматриваете агент (поддержка клиентов, обработка заявок, автоматизация внутренних задач)? Есть ли у вас ограничения по системам или требованиям к безопасности/соответствию?

Какой объём — примерный поток запросов в день/месяц, сколько сотрудников участвует в процессе и какие текущие KPI важно улучшить? Мы разрабатываем кастомные AI‑агенты, интегрируем через API и подбираем архитектуру под задачу; это позволяет автоматизировать рутину, снизить операционные расходы и ускорить обработку 24/7. Стоимость разработки обычно от 50 000 до 300 000 ₽ (без НДС), сопровождение — отдельный продукт. Можете коротко описать процесс или прислать пример типового запроса, чтобы я предложил дальнейшие шаги?' 


router:
{
    "intents": [
        "consult"
    ]
}

 consult: 
 Какой конкретно процесс хотите автоматизировать и какие типовые задачи агент должен решать? Примерные объёмы — запросы/пользователи в сутки и ключевые KPI, которых хотите достичь? Какие системы и данные нужно интегрировать (CRM, база данных, почта и т.п.) и есть ли ограничения по доступу или конфиденциальности?

Мы разрабатываем кастомные AI‑агенты на базе проприетарных LLM и интегрируем их через API, подбирая архитектуру под задачу. Такое решение снижает операционные расходы, автоматизирует рутину, работает 24/7 и ускоряет обработку запросов. Стоимость проектов — от 50 000 до 300 000 ₽ (без НДС); сопровождение после сдачи — отдельный продукт. Можете кратко описать процесс или прислать 3 ключевых требования — предложу варианты подхода.
request:
 «Мы рассматриваем запуск AI-агента, нужен ваш комментарий.»

router:
{
    "intents": [
        "consult"
    ]
}

 consult: 
 Какие конкретные задачи вы хотите поручить агенту 

In [15]:
# save result
df_test.to_excel('result_consult.xlsx', index=False)

## goodbye

In [16]:
row, column = df_test3.shape
index_row = 0
context = " "
# test array
while index_row < row:
    question = extract_request(df_test3, index_row)
    answer, context = neuro_seller(question, context, execution_order, handlers)  # type: ignore
    include_response(df_test3, index_row, answer)
    context = " "
    index_row += 1

request:
 «Ты вообще тупой?»

router:
{
    "intents": [
        "goodbye_hard"
    ]
}

 goodbye: 
  Диалог завершён. Желаю успехов.
request:
 «Читай внимательнее, идиот»


/tmp/ipykernel_58626/2666280072.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Диалог завершён. Желаю успехов.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[index_row, "response"] = response



router:
{
    "intents": [
        "goodbye_hard"
    ]
}

 goodbye: 
  Взаимодействие завершено. Удачи.
request:
 «Ты дебил или прикидываешься?»

router:
{
    "intents": [
        "goodbye_hard"
    ]
}

 goodbye: 
  Диалог завершён. Всего доброго.
request:
 «Закрой рот и делай, что сказали»

router:
{
    "intents": [
        "goodbye_hard"
    ]
}

 goodbye: 
  Диалог завершён. Дальнейшее общение не предусмотрено. Желаю удачи.
request:
 «Мне с такими клоунами разговаривать не о чем»

router:
{
    "intents": [
        "goodbye_hard"
    ]
}

 goodbye: 
  Диалог завершён. Желаю удачи.
request:
 «Вы все там одинаковые, бесполезные»

router:
{
    "intents": [
        "goodbye_hard"
    ]
}

 goodbye: 
  Диалог завершен. Желаю удачи в дальнейшем.
request:
 «Ты кто такой вообще?»

router:
{
    "intents": [
        "consult"
    ]
}

 consult: 
 Я — консультант по AI-решениям в lambda19: мы разрабатываем и интегрируем кастомные AI‑агенты на базе проприетарных LLM через API для автомат

In [17]:
# save result
df_test3.to_excel('result_goodbye.xlsx', index=False)